In [ ]:
# ── Cell 1: Mount Drive ───────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/csao_outputs"


Mounted at /content/drive


In [ ]:
# ── Cell 2: Load all inputs ───────────────────────────────────────────────────

import pandas as pd
import numpy as np

rank_train = pd.read_csv(f"{BASE}/rank_train_data.csv")
rank_test  = pd.read_csv(f"{BASE}/rank_test_data.csv")
items      = pd.read_csv(f"{BASE}/items_clean.csv")
users      = pd.read_csv(f"{BASE}/users.csv")
rests      = pd.read_csv(f"{BASE}/restaurants.csv")
sessions   = pd.read_csv(f"{BASE}/sessions_clean.csv")
cart       = pd.read_csv(f"{BASE}/cart_events_clean.csv")

print(f"rank_train : {len(rank_train):,} rows")
print(f"rank_test  : {len(rank_test):,} rows")
print(f"items      : {len(items):,} rows")
print(f"users      : {len(users):,} rows")
print(f"restaurants: {len(rests):,} rows")
print(f"sessions   : {len(sessions):,} rows")
print(f"cart       : {len(cart):,} rows")

rank_train : 2,133,065 rows
rank_test  : 533,556 rows
items      : 5,575 rows
users      : 10,000 rows
restaurants: 200 rows
sessions   : 94,417 rows
cart       : 195,579 rows


In [ ]:
# ── Cell 3: Build all feature tables ─────────────────────────────────────────

# ── 0) Standardize types ──────────────────────────────────────────────────────
for df in [rank_train, rank_test, items, users, rests, sessions, cart]:
    if "session_id" in df.columns:
        df["session_id"] = df["session_id"].astype(int)

for df in [rank_train, rank_test, cart]:
    if "step_number" in df.columns:
        df["step_number"] = df["step_number"].astype(int)

# ── 1) Cart-state features per (session_id, step_number) ─────────────────────
# Cart state at step N = items added at steps 1 through N (inclusive).
# This matches Stage 1's definition of cart_so_far.
cart_feat = cart.merge(
    items[["item_id", "price", "effective_category"]],
    on="item_id",
    how="left"
)
cart_feat = cart_feat.sort_values(["session_id", "step_number"])

# Cumulative count and value (state AFTER adding item at this step)
cart_feat["cart_item_count"] = cart_feat.groupby("session_id").cumcount() + 1
cart_feat["cart_total_value"] = cart_feat.groupby("session_id")["price"].cumsum()

# Category presence flags
for cat in ["main", "beverage", "dessert", "side"]:
    cart_feat[f"is_{cat}_item"] = (cart_feat["effective_category"] == cat).astype(int)
    cart_feat[f"cart_has_{cat}"] = (
        cart_feat.groupby("session_id")[f"is_{cat}_item"].cummax()
    )

# Last item category at this step
cart_feat["last_item_category"] = cart_feat["effective_category"].fillna("unknown")

cart_state = cart_feat[[
    "session_id", "step_number",
    "cart_item_count", "cart_total_value",
    "cart_has_main", "cart_has_beverage", "cart_has_dessert", "cart_has_side",
    "last_item_category"
]].copy()

# Missing category flags: has main but missing complementary category
cart_state["missing_beverage_flag"] = (
    (cart_state["cart_has_main"] == 1) & (cart_state["cart_has_beverage"] == 0)
).astype(int)
cart_state["missing_dessert_flag"] = (
    (cart_state["cart_has_main"] == 1) & (cart_state["cart_has_dessert"] == 0)
).astype(int)
cart_state["missing_side_flag"] = (
    (cart_state["cart_has_main"] == 1) & (cart_state["cart_has_side"] == 0)
).astype(int)

print(f"cart_state built: {len(cart_state):,} rows")

# ── 2) Session features ───────────────────────────────────────────────────────
sess_cols = ["session_id", "restaurant_id", "meal_time_bucket"]
for col in ["hour_of_day", "is_weekend", "user_id"]:
    if col in sessions.columns:
        sess_cols.append(col)
sess_small = sessions[sess_cols].copy()

# ── 3) Candidate item features ────────────────────────────────────────────────
cand_item_cols = [
    "item_id", "restaurant_id", "price", "popularity_score",
    "historical_attach_rate", "effective_category"
]
if "dish_subtype" in items.columns:
    cand_item_cols.append("dish_subtype")
if "meal_time_suitability" in items.columns:
    cand_item_cols.append("meal_time_suitability")  # kept for FIX 1 computation

items_small = (
    items[cand_item_cols].copy()
    .rename(columns={"item_id": "candidate_item_id"})
)

# ── 4) User features ──────────────────────────────────────────────────────────
user_cols = [
    "user_id", "order_count_90d", "recency_days", "avg_order_value",
    "veg_preference_ratio", "dessert_affinity_score", "beverage_affinity_score",
    "price_sensitivity_score", "user_segment", "city"
]
users_small = users[[c for c in user_cols if c in users.columns]].copy()

# ── 5) Restaurant features ────────────────────────────────────────────────────
rest_cols = [
    "restaurant_id", "city", "cuisine_type", "price_bucket",
    "aggregate_rating", "is_chain", "order_volume_30d", "overall_attach_rate"
]
rests_small = rests[[c for c in rest_cols if c in rests.columns]].copy()

print("All feature tables ready.")

cart_state built: 195,579 rows
All feature tables ready.


In [ ]:
# ── Cell 4: Feature builder function ─────────────────────────────────────────

def build_features(rank_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    df = rank_df.copy()

    # Join session context
    df = df.merge(sess_small, on="session_id", how="left")

    # Join cart state at that step
    df = df.merge(cart_state, on=["session_id", "step_number"], how="left")

    # Join candidate item features
    df = df.merge(items_small, on="candidate_item_id", how="left", suffixes=("", "_cand"))

    # Join user features
    if "user_id" in df.columns:
        df = df.merge(users_small, on="user_id", how="left", suffixes=("", "_user"))

    # Join restaurant features
    df = df.merge(rests_small, on="restaurant_id", how="left", suffixes=("", "_rest"))

    # ── Resolve duplicate columns from merges ─────────────────────────────────
    if "restaurant_id_x" in df.columns:
        df["restaurant_id"] = df["restaurant_id_x"]
    if "restaurant_id_y" in df.columns and "restaurant_id" not in df.columns:
        df["restaurant_id"] = df["restaurant_id_y"]
    for c in ["restaurant_id_x", "restaurant_id_y"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)

    if "meal_time_bucket_x" in df.columns:
        df["meal_time_bucket"] = df["meal_time_bucket_x"]
    elif "meal_time_bucket_y" in df.columns and "meal_time_bucket" not in df.columns:
        df["meal_time_bucket"] = df["meal_time_bucket_y"]
    for c in ["meal_time_bucket_x", "meal_time_bucket_y"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)

    if "city_x" in df.columns:
        df["city"] = df["city_x"]
    for c in ["city_x", "city_y"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)

    # ── Fill numeric NaNs ─────────────────────────────────────────────────────
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # ── FIX 1: meal_time_specificity (replaces binary meal_time_overlap) ──────
    # Problem with old approach: 72% of rows had overlap=1 because LLM tagged
    # most items as "lunch|dinner" and most sessions are lunch/dinner.
    # New approach: score = 1/num_suitable_times if match, else 0.
    # An item tagged only "breakfast" and shown in a breakfast session scores 1.0.
    # An item tagged "breakfast|lunch|dinner|late-night" scores 0.25.
    # This rewards specificity — more informative for the model.
    if "meal_time_bucket" in df.columns and "meal_time_suitability" in df.columns:
        def meal_time_specificity(row):
            suit_str = str(row.get("meal_time_suitability", ""))
            session_mt = str(row.get("meal_time_bucket", "")).lower().strip()
            if not suit_str or suit_str == "nan":
                return 0.0
            tokens = [t.strip().lower() for t in suit_str.split("|") if t.strip()]
            if not tokens or session_mt not in tokens:
                return 0.0
            return 1.0 / len(tokens)  # more specific = higher score

        df["meal_time_specificity"] = df.apply(meal_time_specificity, axis=1)
        # Also keep binary overlap for backwards compatibility / feature importance
        df["meal_time_overlap"] = (df["meal_time_specificity"] > 0).astype(int)
        # Drop raw string — already encoded
        df.drop(columns=["meal_time_suitability"], inplace=True, errors="ignore")
    else:
        df["meal_time_specificity"] = 0.0
        df["meal_time_overlap"] = 0

    # ── FIX 4: step_decay feature ─────────────────────────────────────────────
    # Diagnostic Check 12 showed positive rate drops sharply at later steps:
    # step 1 → 7.7%,  step 2 → 4.4%,  step 3 → 3.3%,  step 4 → 2.4%
    # step_decay = 1/step_number gives the model direct access to this signal.
    df["step_decay"] = 1.0 / df["step_number"].clip(lower=1)

    # ── Interaction features ──────────────────────────────────────────────────
    for col in ["missing_beverage_flag", "missing_dessert_flag", "missing_side_flag",
                "beverage_affinity_score", "dessert_affinity_score",
                "price_sensitivity_score", "price", "cart_total_value"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Candidate category one-hots (from effective_category of candidate item)
    # Note: these are created by get_dummies below — we do NOT manually duplicate them
    # FIX 3: old code created cand_is_* as copies of effective_category_* columns
    # We instead rely on the one-hot columns directly and name them clearly

    # Affinity × missing category (does user like what cart is missing?)
    df["x_missing_bev_aff"] = df["missing_beverage_flag"] * df["beverage_affinity_score"]
    df["x_missing_des_aff"] = df["missing_dessert_flag"]  * df["dessert_affinity_score"]

    # Affinity × candidate category (does user like this type of item?)
    # These will be computed AFTER get_dummies — see post-dummies block below

    # Price interactions
    df["x_price_sens_price"]  = df["price_sensitivity_score"] * df["price"]
    df["x_cart_value_price"]  = df["cart_total_value"] * df["price"]

    # Retrieval × attach rate (strong item in retrieval AND historically popular)
    df["retrieval_x_attach"]     = df["retrieval_score"] * df["historical_attach_rate"]
    df["retrieval_x_popularity"] = df["retrieval_score"] * df["popularity_score"]

    # Missing category × candidate matches × retrieval score (triple signal)
    df["missing_bev_x_retrieval"] = (
        df["missing_beverage_flag"] * df["retrieval_score"]
    )
    df["missing_des_x_retrieval"] = (
        df["missing_dessert_flag"] * df["retrieval_score"]
    )

    # ── One-hot encode categoricals ───────────────────────────────────────────
    # Columns to encode — meal_time_bucket kept for one-hot, raw string dropped after
    cat_cols = []
    for c in ["last_item_category", "effective_category",
              "user_segment", "cuisine_type", "price_bucket"]:
        if c in df.columns:
            cat_cols.append(c)

    # Drop meal_time_bucket string (already encoded into specificity/overlap)
    if "meal_time_bucket" in df.columns:
        df.drop(columns=["meal_time_bucket"], inplace=True, errors="ignore")

    df = pd.get_dummies(df, columns=cat_cols, dummy_na=False)

    # ── Post-dummies: affinity × candidate category interactions ─────────────
    # FIX 3: instead of duplicating effective_category_* as cand_is_*,
    # we compute interactions directly from the one-hot columns
    for cat, aff_col in [("beverage", "beverage_affinity_score"),
                         ("dessert",  "dessert_affinity_score")]:
        ec_col = f"effective_category_{cat}"
        if ec_col in df.columns and aff_col in df.columns:
            df[f"x_cand_{cat[:3]}_aff"] = df[ec_col] * df[aff_col]

    print(f"  [{split_name}] rows: {len(df):,}  cols: {df.shape[1]}")
    return df

In [ ]:
# ── Cell 5: Build train and test feature tables ───────────────────────────────

print("Building train features ...")
train_feat = build_features(rank_train, "train")

print("Building test features ...")
test_feat  = build_features(rank_test, "test")

# ── FIX 2: Align columns between train and test ───────────────────────────────
# get_dummies on each split independently can produce different columns if
# a categorical value appears in one split but not the other.
# Solution: use train as the reference — add missing cols to test as 0,
# drop extra cols from test.

train_cols = train_feat.columns.tolist()
test_cols  = test_feat.columns.tolist()

# Columns in train but missing from test
missing_in_test = [c for c in train_cols if c not in test_cols]
if missing_in_test:
    print(f"\n⚠️  Adding {len(missing_in_test)} columns to test (present in train, missing in test):")
    print(f"   {missing_in_test}")
    for col in missing_in_test:
        test_feat[col] = 0

# Columns in test but not in train — drop them
extra_in_test = [c for c in test_feat.columns if c not in train_cols]
if extra_in_test:
    print(f"\n⚠️  Dropping {len(extra_in_test)} extra columns from test (not in train):")
    print(f"   {extra_in_test}")
    test_feat.drop(columns=extra_in_test, inplace=True)

# Reorder test columns to exactly match train
test_feat = test_feat[train_cols]

print(f"\n✅ Column alignment complete")
print(f"   Train cols: {len(train_feat.columns)}")
print(f"   Test  cols: {len(test_feat.columns)}")
assert list(train_feat.columns) == list(test_feat.columns), "Column mismatch!"
print(f"   ✅ Columns identical between train and test")

Building train features ...
  [train] rows: 2,133,065  cols: 75
Building test features ...
  [test] rows: 533,556  cols: 75

✅ Column alignment complete
   Train cols: 75
   Test  cols: 75
   ✅ Columns identical between train and test


In [ ]:
# ── Cell 6: Sanity checks ─────────────────────────────────────────────────────

print("\n=== Sanity Checks ===")

print(f"\nTrain label rate : {train_feat['label_addon_added'].mean():.4f}")
print(f"Test  label rate : {test_feat['label_addon_added'].mean():.4f}")

print(f"\nTrain meal_time_overlap mean      : {train_feat['meal_time_overlap'].mean():.4f}")
print(f"Train meal_time_specificity mean  : {train_feat['meal_time_specificity'].mean():.4f}")
print(f"(specificity distributes the signal — should have mean < overlap mean)")

print(f"\nstep_decay values: {train_feat['step_decay'].unique()}")

# Verify no cand_is_* duplicates exist
cand_is_cols = [c for c in train_feat.columns if c.startswith("cand_is_")]
print(f"\ncand_is_* columns (should be 0 — FIX 3): {cand_is_cols}")

# Check new interaction features exist
for col in ["retrieval_x_attach", "retrieval_x_popularity",
            "missing_bev_x_retrieval", "missing_des_x_retrieval",
            "meal_time_specificity", "step_decay"]:
    present = col in train_feat.columns
    print(f"  {'✅' if present else '❌'} {col}")

# Positive rate by step (all steps should have > 0 now)
print(f"\nPositive rate by step_number (train):")
print(train_feat.groupby("step_number")["label_addon_added"]
      .agg(["mean", "count"])
      .rename(columns={"mean": "pos_rate", "count": "n_rows"})
      .round(4)
      .to_string())

print(f"\nFinal shapes:")
print(f"  Train: {train_feat.shape}")
print(f"  Test:  {test_feat.shape}")


=== Sanity Checks ===

Train label rate : 0.0379
Test  label rate : 0.0379

Train meal_time_overlap mean      : 0.7237
Train meal_time_specificity mean  : 0.3543
(specificity distributes the signal — should have mean < overlap mean)

step_decay values: [1.         0.5        0.33333333 0.25       0.2       ]

cand_is_* columns (should be 0 — FIX 3): []
  ✅ retrieval_x_attach
  ✅ retrieval_x_popularity
  ✅ missing_bev_x_retrieval
  ✅ missing_des_x_retrieval
  ✅ meal_time_specificity
  ✅ step_decay

Positive rate by step_number (train):
             pos_rate  n_rows
step_number                  
1              0.0367  992143
2              0.0381  624898
3              0.0394  323328
4              0.0408  142290
5              0.0423   50406

Final shapes:
  Train: (2133065, 75)
  Test:  (533556, 75)


In [ ]:
TRAIN_OUT = "lgbm_train_features_v2.csv"
TEST_OUT  = "lgbm_test_features_v2.csv"

train_feat.to_csv(TRAIN_OUT, index=False)
test_feat.to_csv(TEST_OUT,  index=False)

print(f"✅ Saved: {TRAIN_OUT}  rows: {len(train_feat):,}  cols: {train_feat.shape[1]}")
print(f"✅ Saved: {TEST_OUT}   rows: {len(test_feat):,}   cols: {test_feat.shape[1]}")

# Save to Drive
import shutil, os
out_dir = "/content/drive/MyDrive/csao_outputs"
os.makedirs(out_dir, exist_ok=True)
shutil.copy(TRAIN_OUT, os.path.join(out_dir, TRAIN_OUT))
shutil.copy(TEST_OUT,  os.path.join(out_dir, TEST_OUT))
print(f"✅ Both files saved to Drive: {out_dir}")

✅ Saved: lgbm_train_features_v2.csv  rows: 2,133,065  cols: 75
✅ Saved: lgbm_test_features_v2.csv   rows: 533,556   cols: 75
✅ Both files saved to Drive: /content/drive/MyDrive/csao_outputs


In [ ]:
print(f"\nTotal columns: {len(train_feat.columns)}")
print("\nAll column names:")
for col in train_feat.columns:
    print(f"  {col}")


Total columns: 75

All column names:
  session_id
  step_number
  candidate_item_id
  retrieval_score
  src_cooc
  src_ctx
  src_rule
  label_addon_added
  hour_of_day
  is_weekend
  user_id
  cart_item_count
  cart_total_value
  cart_has_main
  cart_has_beverage
  cart_has_dessert
  cart_has_side
  missing_beverage_flag
  missing_dessert_flag
  missing_side_flag
  restaurant_id
  price
  popularity_score
  historical_attach_rate
  dish_subtype
  order_count_90d
  recency_days
  avg_order_value
  veg_preference_ratio
  dessert_affinity_score
  beverage_affinity_score
  price_sensitivity_score
  city
  city_rest
  aggregate_rating
  is_chain
  order_volume_30d
  overall_attach_rate
  meal_time_specificity
  meal_time_overlap
  step_decay
  x_missing_bev_aff
  x_missing_des_aff
  x_price_sens_price
  x_cart_value_price
  retrieval_x_attach
  retrieval_x_popularity
  missing_bev_x_retrieval
  missing_des_x_retrieval
  last_item_category_beverage
  last_item_category_dessert
  last_item_c